# Notebook 13B — Frozen vs Updated XGBoost Reliability

**Question:** Did retraining improve Freddie Mac mortgage credit-risk reliability?

This notebook continues Notebook 13 after the completed Stage 2 fit. It compares the frozen 2015–2017 XGBoost with the updated Policy B XGBoost, using the **same 2023-H1 and 2023-H2 loans**. It does not fit or modify either model.

We report ROC-AUC, Average Precision, Brier Score and absolute risk gap, followed by paired bootstrap uncertainty and calibration charts.

**Scope:** This is a retrospective exploratory comparison, not untouched prospective validation. The January 2023 maturity/availability dates in Notebook 13 were proxies, not confirmed historical publication dates.

## 1. Verify the saved model artifacts

Use the updated model saved in Stage 2 on November 11, 2026. The checks below confirm the original data, preprocessing contract, and both model files have not changed. XGBoost settings are compared individually, treating two missing-value NaNs as equivalent; any substantive mismatch is printed and stops the evaluation.


In [ ]:
from pathlib import Path
import sys, json, hashlib, zipfile
import numpy as np
import pandas as pd
import joblib
import matplotlib.pyplot as plt
from sklearn.metrics import roc_auc_score, average_precision_score, brier_score_loss
from IPython.display import display

ROOT = (Path.cwd().parent if Path.cwd().name.lower() == 'notebooks' else Path.cwd()).resolve()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from src.freddie_features import (REQUIRED_SOURCE_COLUMNS, PREDICTOR_COLUMNS,
    NUMERIC_FEATURES, build_engineered_features, apply_preprocessing_contract)

DATA = ROOT / 'data' / 'processed' / 'Freddie_Mac'
BASE = DATA / 'model_baselines_2015_2017'
RUN_ID = '20261111T161902005765Z'  # Reviewed Stage 2 run; change only if the saved run differs
UPDATE_DIR = DATA / 'notebook_13_updated_xgboost' / RUN_ID
OUTPUT_DIR = DATA / 'notebook_13_retraining_evaluation' / RUN_ID

MODEL_DATA = DATA / 'freddie_modeling_dataset_2015_2023_24m.parquet'
BASE_MANIFEST = BASE / 'model_manifest.json'
FEATURE_CONTRACT = BASE / 'freddie_feature_contract.json'
UPDATE_MANIFEST = UPDATE_DIR / 'updated_model_manifest.json'
SAVED_PREDICTIONS = BASE / 'oot_evaluation' / 'oot_predictions_2018_2023.parquet'
SAVED_BINS = BASE / 'oot_evaluation' / 'frozen_calibration_bins.json'
KEYS = ['SOURCE YEAR', 'LOAN IDENTIFIER']
TARGET = 'TARGET_COMPOSITE_CREDIT_EVENT'

def require(ok, message):
    if not ok:
        raise ValueError(message)

def sha256(path):
    with Path(path).open('rb') as f:
        return hashlib.file_digest(f, 'sha256').hexdigest()

for p in [MODEL_DATA, BASE_MANIFEST, FEATURE_CONTRACT, UPDATE_MANIFEST,
          SAVED_PREDICTIONS, SAVED_BINS]:
    require(p.is_file(), f'Missing required file: {p}')

baseline_info = json.loads(BASE_MANIFEST.read_text(encoding='utf-8'))
update_info = json.loads(UPDATE_MANIFEST.read_text(encoding='utf-8'))
contract = json.loads(FEATURE_CONTRACT.read_text(encoding='utf-8'))
frozen_path = BASE / baseline_info['models']['xgboost']['artifact']
updated_path = UPDATE_DIR / update_info['model_filename']
for p in [frozen_path, updated_path]:
    require(p.is_file(), f'Missing model: {p}')

require(sha256(MODEL_DATA) == baseline_info['input_file_sha256'], 'Modeling data changed.')
require(sha256(FEATURE_CONTRACT) == baseline_info['feature_contract_sha256'], 'Frozen contract changed.')
require(sha256(FEATURE_CONTRACT) == update_info['original_feature_contract_sha256'],
        'Stage 2 used a different feature contract.')
require(sha256(frozen_path) == baseline_info['models']['xgboost']['sha256'] == update_info['frozen_model_sha256'],
        'Frozen model has changed.')
require(sha256(updated_path) == update_info['updated_model_sha256'], 'Updated model has changed.')
require(baseline_info['predictors'] == PREDICTOR_COLUMNS and len(PREDICTOR_COLUMNS) == 29,
        'The 29-predictor definition differs.')
require(contract['training_years'] == [2015, 2016, 2017], 'Contract training years differ.')
require(update_info['primary_training_policy'] == 'B_full_window_individual'
        and update_info['decision_month'] == '2023-01', 'The Stage 2 policy differs.')
require(update_info['sample_rows'] == 285070 and update_info['sample_events'] == 5179,
        'Training sample counts do not match the reviewed results.')
require(update_info['original_fitted_preprocessing_reused'] is True,
        'Stage 2 did not confirm frozen preprocessing reuse.')
require(update_info['evaluation_cohorts'] == ['2023-H1', '2023-H2'],
        'Evaluation cohort list differs.')

frozen_model = joblib.load(frozen_path)
updated_model = joblib.load(updated_path)

# Compare all configured XGBoost settings, treating NaN missing-value defaults as equal.
# A direct dictionary == test may falsely report a change because NaN != NaN.
def equal_setting(a, b):
    if a is None or b is None:
        return a is b
    if isinstance(a, (float, np.floating)) and isinstance(b, (float, np.floating)):
        if np.isnan(a) and np.isnan(b):
            return True
    if isinstance(a, np.ndarray) or isinstance(b, np.ndarray):
        return bool(np.array_equal(a, b, equal_nan=True))
    return bool(a == b)

frozen_params = frozen_model.named_steps['model'].get_params()
updated_params = updated_model.named_steps['model'].get_params()
all_keys = sorted(set(frozen_params) | set(updated_params))
mismatches = [key for key in all_keys
              if key not in frozen_params or key not in updated_params
              or not equal_setting(frozen_params[key], updated_params[key])]
if mismatches:
    print('XGBoost settings requiring review:')
    for key in mismatches:
        print(f'  {key}: frozen={frozen_params.get(key)!r}, updated={updated_params.get(key)!r}')
require(not mismatches, 'XGBoost settings actually differ; review the listed settings before continuing.')
for key, value in baseline_info['models']['xgboost']['selected_params'].items():
    require(equal_setting(frozen_params.get(key), value) and
            equal_setting(updated_params.get(key), value),
            f'Selected baseline setting changed: {key}')
require(update_info['all_original_xgboost_parameters_preserved'] is True,
        'Stage 2 did not confirm original XGBoost parameter preservation.')
print('All XGBoost settings match (including NaN-safe missing-value handling).')
np.testing.assert_array_equal(
    frozen_model.named_steps['prepare'].get_feature_names_out(),
    updated_model.named_steps['prepare'].get_feature_names_out()
)
print('Model and data fingerprints verified.')
print('Primary training policy:', update_info['primary_training_policy'])
print('Training loans:', update_info['sample_rows'], '| positive events:', update_info['sample_events'])


## 2. Create the same evaluation population for both models

We apply the original 2015–2017 feature contract to 2023 originations. We then compare newly generated frozen-model predictions with those saved in Notebook 11. Matching those predictions is an important reproducibility check.

No new training or category fitting is performed.

In [ ]:
raw = pd.read_parquet(MODEL_DATA, columns=list(dict.fromkeys([*REQUIRED_SOURCE_COLUMNS, 'HORIZON_MONTHS'])))
raw['SOURCE YEAR'] = pd.to_numeric(raw['SOURCE YEAR'], errors='raise').astype(int)
raw['LOAN IDENTIFIER'] = raw['LOAN IDENTIFIER'].astype('string')
require(len(raw) == 437668 and not raw.duplicated(KEYS).any(), 'Validated modeling data totals differ.')
raw_2023 = raw.loc[raw['SOURCE YEAR'].eq(2023)].copy()
require(raw_2023['HORIZON_MONTHS'].eq(24).all(), 'Unexpected label horizon.')
del raw

engineered = build_engineered_features(raw_2023)
prepared = apply_preprocessing_contract(engineered, contract)
require(not prepared[PREDICTOR_COLUMNS].isna().any().any(), 'Missing model predictors.')
require(np.isfinite(prepared[NUMERIC_FEATURES].to_numpy(dtype=float)).all(),
        'Infinite or missing numeric predictor.')
X = prepared.loc[:, PREDICTOR_COLUMNS]
evaluation = prepared.loc[:, KEYS + ['ORIGINATION QUARTER', TARGET]].copy()
q = pd.to_numeric(evaluation['ORIGINATION QUARTER'].astype('string').str.strip().str.removeprefix('Q'),
                  errors='raise').astype(int)
require(q.isin([1, 2, 3, 4]).all(), 'Invalid quarter value.')
evaluation['period'] = np.where(q.le(2), '2023-H1', '2023-H2')
evaluation = evaluation.rename(columns={TARGET: 'y_true'}).drop(columns='ORIGINATION QUARTER')
require(not evaluation.duplicated(KEYS).any(), 'Duplicate evaluation keys.')
evaluation['p_frozen'] = frozen_model.predict_proba(X)[:, 1]
evaluation['p_updated'] = updated_model.predict_proba(X)[:, 1]

saved = pd.read_parquet(SAVED_PREDICTIONS)
require(set(KEYS + ['y_true', 'p_xgboost', 'half_year']).issubset(saved.columns),
        'Saved Notebook 11 prediction columns differ.')
saved['SOURCE YEAR'] = pd.to_numeric(saved['SOURCE YEAR'], errors='raise').astype(int)
saved['LOAN IDENTIFIER'] = saved['LOAN IDENTIFIER'].astype('string')
saved = saved.loc[saved['SOURCE YEAR'].eq(2023),
                  KEYS + ['y_true', 'p_xgboost', 'half_year']].copy()
require(not saved.duplicated(KEYS).any(), 'Duplicate saved prediction keys.')
joined = evaluation.merge(saved, how='outer', on=KEYS, validate='one_to_one',
                          indicator=True, suffixes=('', '_saved'))
require(joined['_merge'].eq('both').all(), '2023 loan keys do not match saved Notebook 11 predictions.')
require(joined['y_true'].eq(joined['y_true_saved']).all() and
        joined['period'].eq(joined['half_year']).all(),
        '2023 outcomes or cohort names differ from Notebook 11.')
np.testing.assert_allclose(joined['p_frozen'], joined['p_xgboost'], rtol=1e-6, atol=1e-8,
                           err_msg='Frozen XGBoost scores do not reproduce Notebook 11.')
evaluation = joined[KEYS + ['period', 'y_true', 'p_frozen', 'p_updated']].copy()
evaluation = evaluation.sort_values(KEYS).reset_index(drop=True)
for col in ['p_frozen', 'p_updated']:
    require(np.isfinite(evaluation[col]).all() and evaluation[col].between(0, 1).all(),
            f'Invalid probabilities: {col}')

cohort_summary = evaluation.groupby('period', sort=True).agg(
    loans=('y_true', 'size'), events=('y_true', 'sum')).reset_index()
expected = {'2023-H1': (24757, 460), '2023-H2': (24744, 416)}
require(set(cohort_summary['period']) == set(expected), 'Unexpected evaluation cohorts.')
for row in cohort_summary.itertuples(index=False):
    require((row.loans, row.events) == expected[row.period], f'Unexpected counts for {row.period}.')
require(len(evaluation) == 49501 and int(evaluation['y_true'].sum()) == 876,
        'Combined 2023 population differs.')
display(cohort_summary)
print('Total evaluation loans:', len(evaluation), '| positive events:', int(evaluation['y_true'].sum()))
print('Frozen model reproduced Notebook 11 predictions on the same 2023 loans.')


## 3. Compare the model performance

- **ROC-AUC**: ability to rank positive loans ahead of negative loans. Higher is better.
- **Average Precision**: ranking quality across precision/recall tradeoffs. Higher is better.
- **Brier Score**: accuracy of probability estimates. Lower is better.
- **Absolute Risk Gap**: absolute difference between actual event rate and average predicted risk. Lower is better.

For each metric, a **positive improvement** in the comparison table means the updated model is better. We report 2023-H1, 2023-H2 and the combined 2023 sample separately.

In [ ]:
def metrics(y, p):
    return {'roc_auc': float(roc_auc_score(y, p)),
            'average_precision': float(average_precision_score(y, p)),
            'brier': float(brier_score_loss(y, p)),
            'risk_gap': float(y.mean() - p.mean()),
            'absolute_risk_gap': float(abs(y.mean() - p.mean())),
            'mean_predicted_risk': float(p.mean())}

METRICS = ['roc_auc', 'average_precision', 'brier', 'absolute_risk_gap']
HIGHER_BETTER = {'roc_auc', 'average_precision'}
groups = {'2023-H1': evaluation.loc[evaluation['period'].eq('2023-H1')],
          '2023-H2': evaluation.loc[evaluation['period'].eq('2023-H2')],
          '2023-all': evaluation}
metric_rows, comparison_rows = [], []
for period, data in groups.items():
    y = data['y_true'].to_numpy(dtype=int)
    require(np.unique(y).size == 2, f'{period}: both classes are required.')
    summary = {}
    for model in ['frozen', 'updated']:
        p = data[f'p_{model}'].to_numpy(dtype=float)
        summary[model] = metrics(y, p)
        metric_rows.append({'period': period, 'model': model, 'loans': len(y),
                            'events': int(y.sum()), 'prevalence': float(y.mean()),
                            **summary[model]})
    for metric in METRICS:
        f = summary['frozen'][metric]
        u = summary['updated'][metric]
        improvement = u - f if metric in HIGHER_BETTER else f - u
        comparison_rows.append({'period': period, 'metric': metric,
                                'frozen': f, 'updated': u,
                                'improvement': improvement,
                                'interpretation': ('updated_better' if improvement > 0 else
                                                   'updated_worse' if improvement < 0 else 'equal')})
model_metrics = pd.DataFrame(metric_rows)
point_comparison = pd.DataFrame(comparison_rows)
display(model_metrics.round(6))
display(point_comparison.round(6))
print('Positive improvement favors the updated model. Results remain exploratory.')


## 4. Check uncertainty using paired bootstrap samples

We resample the **same loan indices** for both models in each draw. For every cohort and the full 2023 set, we calculate 1,000 resampled differences and summarize them with a 95% percentile interval.

This measures evaluation-sample uncertainty conditional on the two trained models. It does not capture model-training uncertainty or shared economic shocks.

In [ ]:
BOOTSTRAP_DRAWS = 1000
rng = np.random.default_rng(42)
interval_rows = []
for period, data in groups.items():
    y = data['y_true'].to_numpy(dtype=int)
    pf = data['p_frozen'].to_numpy(dtype=float)
    pu = data['p_updated'].to_numpy(dtype=float)
    draws = {m: [] for m in METRICS}
    invalid_draws = 0
    for _ in range(BOOTSTRAP_DRAWS):
        ix = rng.integers(0, len(y), size=len(y))
        sampled_y = y[ix]
        if np.unique(sampled_y).size < 2:
            invalid_draws += 1
            continue
        frozen_result = metrics(sampled_y, pf[ix])
        updated_result = metrics(sampled_y, pu[ix])
        for metric in METRICS:
            f, u = frozen_result[metric], updated_result[metric]
            draws[metric].append(u - f if metric in HIGHER_BETTER else f - u)
    for metric in METRICS:
        vals = np.asarray(draws[metric])
        require(len(vals) > 0, f'No valid bootstrap draws for {period}, {metric}.')
        lo, hi = np.quantile(vals, [0.025, 0.975])
        interval_rows.append({'period': period, 'metric': metric,
                              'ci_low': float(lo), 'ci_high': float(hi),
                              'valid_draws': len(vals), 'invalid_one_class_draws': invalid_draws,
                              'direction': ('favors_updated' if lo > 0 else
                                            'favors_frozen' if hi < 0 else 'uncertain')})
    print('Completed paired bootstrap:', period, '| invalid one-class draws:', invalid_draws)
intervals = pd.DataFrame(interval_rows)
paired_results = point_comparison.merge(intervals, on=['period', 'metric'], validate='one_to_one')
display(paired_results.round(6))
print('A confidence interval entirely above 0 favors the updated model; below 0 favors frozen.')


## 5. Inspect calibration using the original development bins

Both models use the XGBoost prediction boundaries saved in Notebook 11. We do not refit bins using 2023 data. The plots compare observed event rates against average predicted probabilities within each occupied bin.

In [ ]:
bins_info = json.loads(SAVED_BINS.read_text(encoding='utf-8'))
require('xgboost' in bins_info, 'Missing XGBoost calibration bins.')
inner_edges = np.asarray(bins_info['xgboost'], dtype=float)
require(len(inner_edges) >= 1 and np.isfinite(inner_edges).all() and
        np.all(np.diff(inner_edges) > 0), 'Invalid saved calibration boundaries.')

calibration_rows = []
calibration_figures = {}
for period, data in groups.items():
    fig, ax = plt.subplots(figsize=(7, 5))
    for model in ['frozen', 'updated']:
        probs = data[f'p_{model}'].to_numpy(dtype=float)
        labels = data['y_true'].to_numpy(dtype=int)
        indices = np.searchsorted(inner_edges, probs, side='right')
        xs, ys = [], []
        for bin_num in range(len(inner_edges) + 1):
            in_bin = indices == bin_num
            n = int(in_bin.sum())
            if n == 0:
                continue
            predicted = float(probs[in_bin].mean())
            actual = float(labels[in_bin].mean())
            calibration_rows.append({'period': period, 'model': model,
                                     'bin_index': bin_num + 1, 'loans': n,
                                     'events': int(labels[in_bin].sum()),
                                     'mean_predicted_risk': predicted,
                                     'observed_event_rate': actual})
            xs.append(predicted)
            ys.append(actual)
        ax.plot(xs, ys, marker='o', label=model.capitalize())
    upper = max(float(data[['p_frozen', 'p_updated']].to_numpy().max()),
                float(data['y_true'].mean()), 0.01)
    # Reference line expresses perfect calibration.
    ax.plot([0, upper], [0, upper], linestyle='--', linewidth=1, color='gray', label='Perfect calibration')
    ax.set_title(f'Calibration: {period}')
    ax.set_xlabel('Mean predicted risk')
    ax.set_ylabel('Observed event rate')
    ax.legend()
    ax.grid(alpha=0.2)
    plt.tight_layout()
    calibration_figures[period] = fig
    plt.show()

calibration_detail = pd.DataFrame(calibration_rows)
checks = calibration_detail.groupby(['period', 'model'])['loans'].sum()
for period, data in groups.items():
    for model in ['frozen', 'updated']:
        require(int(checks.loc[(period, model)]) == len(data), 'Calibration bin coverage differs.')
display(calibration_detail.round(6))


## 6. Save an aggregate review bundle

This saves only summary tables, calibration figures, input/model fingerprints and a small ZIP for review. **Do not commit raw Freddie Mac loans, loan-level predictions, or model joblib files to GitHub.** The saved model files from Stage 2 remain untouched.

In [ ]:
require(sha256(frozen_path) == baseline_info['models']['xgboost']['sha256'],
        'The frozen model changed during evaluation.')
require(sha256(updated_path) == update_info['updated_model_sha256'],
        'The updated model changed during evaluation.')
require(not OUTPUT_DIR.exists(),
        f'An evaluation folder already exists: {OUTPUT_DIR}. Review it instead of overwriting.')
OUTPUT_DIR.mkdir(parents=True)

files = {'evaluation_population_summary.csv': cohort_summary,
         'frozen_updated_metrics.csv': model_metrics,
         'frozen_updated_point_comparison.csv': point_comparison,
         'frozen_updated_paired_intervals.csv': paired_results,
         'calibration_bin_comparison.csv': calibration_detail}
for name, frame in files.items():
    frame.to_csv(OUTPUT_DIR / name, index=False)

for period, fig in calibration_figures.items():
    fig.savefig(OUTPUT_DIR / f'calibration_{period}.png', dpi=160, bbox_inches='tight')

summary_manifest = {
    'stage': 'Notebook 13B: frozen versus updated XGBoost comparison',
    'analysis_type': 'retrospective_exploratory',
    'primary_training_policy': update_info['primary_training_policy'],
    'decision_month': update_info['decision_month'],
    'training_loans': update_info['sample_rows'],
    'training_events': update_info['sample_events'],
    'evaluation_periods': ['2023-H1', '2023-H2'],
    'evaluation_loans': int(len(evaluation)),
    'evaluation_events': int(evaluation['y_true'].sum()),
    'bootstrap_draws': BOOTSTRAP_DRAWS,
    'frozen_model_sha256': sha256(frozen_path),
    'updated_model_sha256': sha256(updated_path),
    'frozen_feature_contract_sha256': sha256(FEATURE_CONTRACT),
    'model_data_sha256': sha256(MODEL_DATA),
    'historical_publication_dates_verified': False,
    'untouched_independent_evaluation': False,
    'limitations': [
        'Historical data release dates were approximated by availability proxies.',
        'Training membership was derived from a retrospectively eligible population.',
        '2023 outcomes were previously reviewed; this is not untouched prospective validation.',
        'Bootstrap intervals omit model-fitting uncertainty and correlated macroeconomic shocks.',
        'No validated economic cost threshold or retraining-trigger rule is estimated here.'
    ]
}
(OUTPUT_DIR / 'evaluation_manifest.json').write_text(
    json.dumps(summary_manifest, indent=2), encoding='utf-8')

review_zip = OUTPUT_DIR / 'notebook_13b_stage3_review.zip'
with zipfile.ZipFile(review_zip, 'w', compression=zipfile.ZIP_DEFLATED) as archive:
    for name in [*files, 'evaluation_manifest.json']:
        archive.write(OUTPUT_DIR / name, arcname=name)
    for period in groups:
        name = f'calibration_{period}.png'
        archive.write(OUTPUT_DIR / name, arcname=name)
with zipfile.ZipFile(review_zip) as archive:
    require(archive.testzip() is None, 'Review ZIP failed integrity check.')
print('Evaluation summary saved locally:', OUTPUT_DIR)
print('Share this aggregate review ZIP:', review_zip)
print('Both models remain unchanged. No raw loan-level records were exported.')


## Review gate

Share `notebook_13b_stage3_review.zip` for verification. Before drawing a conclusion, review whether the updated model improves **both discrimination and probability reliability**, or whether the metrics trade off. Review cohort-specific behavior and the paired intervals, not just the combined 2023 values.

**Do not push the executed notebook to GitHub until the results are reviewed.** Keep the original baseline models, the Stage 2 updated model and all original data unchanged. External Fannie Mae reliability remains a separate later milestone.